## layernorm backward test case

In [192]:
import torch
import torch.nn as nn

In [312]:
#init backward case 1
x = torch.tensor([[[   0.8351,  -52.3470, -120.6479,   79.4512],
         [ -53.9348,  116.3453,  -26.2333, -111.2108],
         [  95.6808, -122.4990, -139.4510,   52.4164]],

        [[ -48.4344,  102.0699,  -32.5717,  -21.7057],
         [   1.3376,   85.8685,  158.5985,   85.5139],
         [ -41.6447, -225.0275,  -35.9537,  -88.3156]]], requires_grad=True)
weight = torch.tensor([16.4667, 10.2935, 11.0739, 12.7998])
bias = torch.tensor([-1.7650, -1.4063,  1.6240,  0.3662])
print(x)

tensor([[[   0.8351,  -52.3470, -120.6479,   79.4512],
         [ -53.9348,  116.3453,  -26.2333, -111.2108],
         [  95.6808, -122.4990, -139.4510,   52.4164]],

        [[ -48.4344,  102.0699,  -32.5717,  -21.7057],
         [   1.3376,   85.8685,  158.5985,   85.5139],
         [ -41.6447, -225.0275,  -35.9537,  -88.3156]]], requires_grad=True)


In [313]:
# pytorch grad
ln = nn.LayerNorm(4)
#[16.4667, 10.2935, 11.0739, 12.7998]
ln.weight = torch.nn.Parameter(weight)
#[-1.7650, -1.4063,  1.6240,  0.3662]
ln.bias = torch.nn.Parameter(bias)
print(ln.weight)
print(ln.bias)
y = ln(x)
y.retain_grad()

Parameter containing:
tensor([16.4667, 10.2935, 11.0739, 12.7998], requires_grad=True)
Parameter containing:
tensor([-1.7650, -1.4063,  1.6240,  0.3662], requires_grad=True)


In [314]:
# diff loss for diff dy
#loss = y.sum()
loss = y.exp().sum()

In [315]:
loss.backward()
print(loss)
print(y.grad)
print(ln.weight.grad)
print(ln.bias.grad)
print(x.grad)
x.grad.zero_()
pass

tensor(1.8225e+08, grad_fn=<SumBackward0>)
tensor([[[3.7833e+01, 4.0641e-03, 2.0205e-06, 8.8659e+07],
         [1.7052e-04, 3.9447e+06, 1.8895e+00, 1.0632e-06],
         [6.0932e+07, 2.1885e-05, 3.6637e-05, 3.0876e+04]],

        [[2.8763e-07, 1.0808e+07, 1.2529e-02, 1.4311e-02],
         [5.8303e-12, 4.2980e-01, 1.7804e+07, 2.6734e+00],
         [3.1208e+04, 8.4363e-09, 4.0009e+04, 7.0111e+00]]])
tensor([7.2884e+07, 2.4841e+07, 2.4263e+07, 1.2425e+08])
tensor([60963540., 14752580., 17843962., 88689384.])
tensor([[[-5.6523e+06, -1.7120e+06,  3.3484e+06,  4.0160e+06],
         [-3.9165e+04,  4.8587e+04, -1.0371e+05,  9.4289e+04],
         [ 3.7918e+06,  2.0043e+05,  6.7229e+05, -4.6645e+06]],

        [[ 1.7716e+05,  3.5280e+04, -3.3924e+04, -1.7852e+05],
         [ 8.7831e+05, -9.5110e+05,  1.0162e+06, -9.4342e+05],
         [ 1.8242e+03,  8.9646e+02,  7.1465e+02, -3.4353e+03]]])


In [308]:
#calculate
x1 = x.detach().clone()
print(x1)

tensor([[[   0.8351,  -52.3470, -120.6479,   79.4512],
         [ -53.9348,  116.3453,  -26.2333, -111.2108],
         [  95.6808, -122.4990, -139.4510,   52.4164]],

        [[ -48.4344,  102.0699,  -32.5717,  -21.7057],
         [   1.3376,   85.8685,  158.5985,   85.5139],
         [ -41.6447, -225.0275,  -35.9537,  -88.3156]]])


In [316]:
### print(x1)
rstd = x1.std(-1,keepdim=True)
print(x1.mean(-1,keepdim=True))
print(x1 - x1.mean(-1,keepdim=True))
rstd =1.0 / (x1.std(-1,keepdim=True))
print(rstd)
x_norm = (x1 - x1.mean(-1,keepdim=True)) * rstd

tensor([[[-23.1772],
         [-18.7584],
         [-28.4632]],

        [[ -0.1605],
         [ 82.8296],
         [-97.7354]]])
tensor([[[  24.0123,  -29.1698,  -97.4707,  102.6284],
         [ -35.1764,  135.1037,   -7.4749,  -92.4524],
         [ 124.1440,  -94.0358, -110.9878,   80.8796]],

        [[ -48.2739,  102.2304,  -32.4112,  -21.5452],
         [ -81.4920,    3.0389,   75.7689,    2.6843],
         [  56.0907, -127.2921,   61.7817,    9.4198]]])
tensor([[[0.0118],
         [0.0103],
         [0.0083]],

        [[0.0145],
         [0.0156],
         [0.0114]]])


In [317]:
# our method pytorch version... 
# its accuracy is extremely poor;
dy = y.grad.detach().clone()
print(dy)
#dy_dx[0][i]
for i in range(4):
    gamma = ln.weight.detach().clone()
    term1 = gamma[i] 
    term2 = gamma.dot(dy[0][0]) / 4
    term3 = x_norm[0,0,i] * (gamma.dot(x_norm[0,0] * dy[0][0])) / 4
    print(rstd[0][0]*(term1 - term2 -term3))


tensor([[[3.7833e+01, 4.0641e-03, 2.0205e-06, 8.8659e+07],
         [1.7052e-04, 3.9447e+06, 1.8895e+00, 1.0632e-06],
         [6.0932e+07, 2.1885e-05, 3.6637e-05, 3.0876e+04]],

        [[2.8763e-07, 1.0808e+07, 1.2529e-02, 1.4311e-02],
         [5.8303e-12, 4.2980e-01, 1.7804e+07, 2.6734e+00],
         [3.1208e+04, 8.4363e-09, 4.0009e+04, 7.0111e+00]]])
tensor([-4509847.5000])
tensor([-1950580.3750])
tensor([1336244.8750])
tensor([-8293069.5000])
